# R-Multiple-Based Exit Rules on SPY
## Strategy Brief
This strategy involves trading SPY using R-Multiple-Based Exit Rules, which focus on risk management by setting predefined risk multiples for exits. The signal is generated based on a simple moving average crossover, and trades are executed with a focus on maintaining a favorable risk-reward ratio. The logic involves entering trades when a short-term moving average crosses above a long-term moving average and exiting based on predefined R-multiples. The results aim to improve risk-adjusted returns compared to a simple buy-and-hold strategy.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we set up the trading environment and define the parameters for our strategy. These parameters include the moving average periods and the R-multiple values for exit rules.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
SHORT_MA_PERIOD = 50
LONG_MA_PERIOD = 200
R_MULTIPLE = 2
RISK_PER_TRADE = 0.01

## PHASE 2 - Data Exploration
We download historical price data for SPY from Yahoo Finance, calculate the moving averages, and visualize them to understand the market context.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate moving averages
data['Short_MA'] = data['Close'].rolling(window=SHORT_MA_PERIOD).mean()
data['Long_MA'] = data['Close'].rolling(window=LONG_MA_PERIOD).mean()

# Plot
data[['Close', 'Short_MA', 'Long_MA']].plot(figsize=(14, 7))
plt.title('SPY Price with Moving Averages')
plt.show()

## PHASE 3 - Strategy Engineering
We define the trading signals and the entry/exit logic based on moving average crossovers and R-Multiple-Based Exit Rules.

In [ ]:
# Generate signals
data['Signal'] = 0
data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1
data.loc[data['Short_MA'] < data['Long_MA'], 'Signal'] = -1

# Entry/Exit logic
data['Position'] = data['Signal'].shift(1)
data['Entry_Price'] = data['Close'].where(data['Position'] == 1)
data['Exit_Price'] = data['Close'].where(data['Position'] == -1)

# Calculate R-multiple exits
data['Stop_Loss'] = data['Entry_Price'] * (1 - RISK_PER_TRADE)
data['Take_Profit'] = data['Entry_Price'] * (1 + R_MULTIPLE * RISK_PER_TRADE)

## PHASE 4 - Coding & Backtesting
We implement the backtesting logic to evaluate the strategy's performance over historical data.

In [ ]:
# Calculate daily returns
data['Market_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = data['Market_Return'] * data['Position'].shift(1)

# Calculate equity curve
data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()

# Plot equity curve
data['Equity_Curve'].plot(figsize=(14, 7))
plt.title('Strategy Equity Curve')
plt.show()

## PHASE 5 - Performance Evaluation
We evaluate the strategy's performance using key metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown, and compare it to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(data):
    cagr = (data['Equity_Curve'].iloc[-1] ** (1 / ((data.index[-1] - data.index[0]).days / 365.25))) - 1
    sharpe_ratio = data['Strategy_Return'].mean() / data['Strategy_Return'].std() * np.sqrt(252)
    downside_std = data[data['Strategy_Return'] < 0]['Strategy_Return'].std()
    sortino_ratio = data['Strategy_Return'].mean() / downside_std * np.sqrt(252)
    max_drawdown = (data['Equity_Curve'].cummax() - data['Equity_Curve']).max()
    calmar_ratio = cagr / max_drawdown
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown = calculate_performance_metrics(data)

# Buy-and-Hold comparison
data['Buy_Hold'] = (1 + data['Market_Return']).cumprod()
bh_cagr = (data['Buy_Hold'].iloc[-1] ** (1 / ((data.index[-1] - data.index[0]).days / 365.25))) - 1

# Display results
performance_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown],
    'Buy & Hold': [bh_cagr, np.nan, np.nan, np.nan, np.nan]
})
print(performance_df)

## PHASE 6 - Deploy & Monitor
We create a function to download the latest data, compute today's signal, and print the current position.

In [ ]:
def get_latest_signal():
    latest_data = yf.download('SPY', period='60d')
    latest_data['Short_MA'] = latest_data['Close'].rolling(window=SHORT_MA_PERIOD).mean()
    latest_data['Long_MA'] = latest_data['Close'].rolling(window=LONG_MA_PERIOD).mean()
    latest_signal = 0
    if latest_data['Short_MA'].iloc[-1] > latest_data['Long_MA'].iloc[-1]:
        latest_signal = 1
    elif latest_data['Short_MA'].iloc[-1] < latest_data['Long_MA'].iloc[-1]:
        latest_signal = -1
    print(f"Today's Signal: {'Buy' if latest_signal == 1 else 'Sell' if latest_signal == -1 else 'Hold'}")

get_latest_signal()